# RC Parameter Identification Report

This notebook demonstrates the full identification-to-report workflow:

1. Generate synthetic measurement data from a known 2R2C model
2. Run cross-validation to identify RC parameters
3. Build an `IdentificationReport` with per-room quality metrics
4. Visualize predicted vs measured temperatures and residuals
5. Use `QualityMonitor` for auto-reidentification detection

In [ ]:
import numpy as np

from pumpahead.cross_validation import cross_validate
from pumpahead.identification_report import (
    IdentificationReport,
    QualityMonitor,
    plot_predicted_vs_measured,
    plot_residuals,
    plot_rmse_over_time,
)
from pumpahead.identifier import RCIdentifier
from pumpahead.model import ModelOrder, RCModel, RCParams

## 1. Define True Parameters and Generate Synthetic Data

We use known 2R2C parameters to generate ground-truth temperature data.
Cyclic heating (4-hour ON/OFF blocks) ensures both fast (C_air) and slow
(C_slab) thermal modes are excited.

In [ ]:
# True parameters for a 20 m^2 room
true_params = RCParams(
    C_air=60_000,  # J/K
    C_slab=3_250_000,  # J/K
    R_sf=0.01,  # K/W
    R_env=0.03,  # K/W
    f_conv=0.6,
    f_rad=0.4,
    T_ground=10.0,
    has_split=False,
)

model = RCModel(true_params, ModelOrder.TWO, dt=60.0)
print(f"Model order: {model.order.name}")
print(f"States: {model.state_names}")
print(f"Inputs: {model.n_inputs}, Disturbances: {model.n_disturbances}")

In [ ]:
# Generate 4 days of synthetic data (5760 steps at dt=60s)
n_steps = 5760
u_seq = np.zeros((n_steps, 1))
for i in range(n_steps):
    block = (i // 240) % 2  # 4-hour ON/OFF blocks
    u_seq[i, 0] = 0.0 if block == 0 else 1500.0  # W

d_seq = np.zeros((n_steps, 2))
d_seq[:, 0] = 5.0  # constant T_out = 5 degC

x0 = np.array([5.0, 5.0])  # start at outdoor temperature
trajectory = model.predict(x0, u_seq, d_seq)
T_room = trajectory[1:, 0]  # T_air at each step

print(f"Data length: {len(T_room)} steps ({len(T_room) / 1440:.1f} days)")
print(f"T_room range: [{T_room.min():.2f}, {T_room.max():.2f}] degC")

## 2. Run Cross-Validation

Cross-validation splits the data into train (70%) and test (30%) segments,
identifies RC parameters on the training set, and evaluates prediction
accuracy on both sets at multiple horizons (6h, 12h, 24h).

In [ ]:
# Configure identifier with informed bounds (near true values)
identifier = RCIdentifier(
    ModelOrder.TWO,
    dt=60.0,
    n_starts=20,
    seed=42,
    bounds={
        "R_sf": (0.005, 0.02),
        "R_env": (0.01, 0.06),
        "C_air": (30_000.0, 120_000.0),
        "C_slab": (1_500_000.0, 6_000_000.0),
    },
)

cv_result = cross_validate(identifier, u_seq, d_seq, T_room)

print(f"Train RMSE: {cv_result.train_rmse:.4f} degC")
print(f"Test  RMSE: {cv_result.test_rmse:.4f} degC")
print(f"Overfitting: {cv_result.is_overfitting}")
print()
for h in cv_result.horizons:
    test_str = f"{h.test_rmse:.4f}" if h.test_rmse is not None else "N/A"
    print(f"  {h.horizon_hours:.0f}h — train: {h.train_rmse:.4f}, test: {test_str}")

## 3. Build Identification Report

The `IdentificationReport.from_cv_results()` factory recomputes predicted
temperatures on the test segment and produces a per-room quality report.

In [ ]:
cv_results = {"living_room": cv_result}
data = {"living_room": (u_seq, d_seq, T_room)}

report = IdentificationReport.from_cv_results(cv_results, data)

print(f"Report created at: {report.created_at}")
print(f"All rooms passed: {report.all_passed}")
print(f"Failed rooms: {report.failed_rooms}")
print()

for room in report.rooms:
    print(f"--- {room.room_name} ---")
    print(f"  RMSE: {room.rmse:.4f} degC")
    print(f"  Threshold: {room.rmse_threshold:.2f} degC")
    print(f"  Passed: {room.passed}")
    print(f"  Identified R_sf: {room.params.R_sf:.6f} K/W")
    print(f"  Identified R_env: {room.params.R_env}")
    print(f"  Identified C_air: {room.params.C_air:.0f} J/K")
    print(f"  Identified C_slab: {room.params.C_slab:.0f} J/K")

## 4. Visualization

### 4a. Predicted vs Measured Temperature

In [ ]:
room = report.rooms[0]
fig = plot_predicted_vs_measured(
    room.T_predicted,
    room.T_measured,
    room_name=room.room_name,
)
fig  # display in notebook

### 4b. Prediction Residuals

In [ ]:
fig_res = plot_residuals(
    room.T_predicted,
    room.T_measured,
    room_name=room.room_name,
)
fig_res

## 5. QualityMonitor — Auto-Reidentification

The `QualityMonitor` tracks RMSE over time and flags rooms where
prediction quality has degraded beyond the threshold.

In [ ]:
monitor = QualityMonitor(rmse_threshold=0.5)

# Simulate RMSE measurements over time
rmse_values = [0.15, 0.18, 0.22, 0.30, 0.35, 0.42, 0.55, 0.61]
for rmse in rmse_values:
    monitor.update("living_room", rmse)
    flag = monitor.needs_reidentification("living_room")
    print(f"RMSE={rmse:.2f}  needs_reidentification={flag}")

print(f"\nFull history: {monitor.get_history('living_room')}")

In [ ]:
fig_rmse = plot_rmse_over_time(
    monitor.get_history("living_room"),
    room_name="living_room",
    threshold=monitor.threshold,
)
fig_rmse

In [ ]:
# After reidentification, reset the monitor
monitor.reset("living_room")
print(f"After reset: history={monitor.get_history('living_room')}")
print(f"needs_reidentification={monitor.needs_reidentification('living_room')}")